# 15 -- LSTM + Transformer FinalShot

**Один оставшийся submission -- notebook сам выбирает один финальный stack и сохраняет только его как основной результат.**

Известные public endpoints:

```text
LSTM        = 1.6506631932
Transformer = 1.6509854083
```

Standalone Transformer немного хуже public, но лучше на последней temporal CV. Поэтому шанс ансамбля есть, если ошибки моделей не полностью совпадают.

## Главная идея FinalShot

Сначала строится максимально надежный **public-informed global log-space blend**.

Это сильнее обычного OOF-blend, потому что мы уже знаем public RMSLE двух концов линии:

```text
w = 0 -> точный public LSTM score
w = 1 -> точный public Transformer score
```

И знаем сами February predictions обеих моделей.

Дальше OOF используется не для того, чтобы проигнорировать leaderboard signal, а для проверки:

- насколько residuals моделей различаются;
- насколько blend weight стабилен во времени;
- есть ли уверенный gain от magnitude-aware adaptation;
- есть ли уверенный gain от маленького residual Ridge.

**Adaptive stack разрешается заменить global blend только если он стабильно выигрывает expanding temporal meta-CV.**

Иначе финальным остается public-informed global blend.

Это специально консервативная стратегия после неудачного MegaStack.

## 0. Что нужно

Обязательно:

```text
models/lstm_hurdle_v4_expanded_es/oof_best_epoch.parquet
models/transformer_hurdle_v1/oof_best_epoch.parquet

submissions/lstm_hurdle_v4_expanded_es_3seed.csv
submissions/transformer_hurdle_v1_3seed.csv
```

Опционально для residual Ridge:

```text
models/lstm_hurdle_v4_expanded_es/february_seed_components.parquet

models/transformer_hurdle_v1/
    inference_seed_42.npz
    inference_seed_143.npz
    inference_seed_2026.npz
```

Если components нет -- notebook все равно полностью считает основной FinalShot через `pred_log`.

На выходе основной файл всегда один:

```text
submissions/lstm_transformer_finalshot.csv
```

Диагностические альтернативы **не сохраняются как submission**, чтобы не было соблазна выбирать руками после OOF.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026

/content/drive/MyDrive/Colab-Notebooks/E-CUP-2026


In [5]:
from pathlib import Path
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyarrow") is None:
    print("pyarrow not found -- installing")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyarrow",
    ])

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_project_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data").exists() and (path / "models").exists():
            return path

    return Path.cwd()


PROJECT_ROOT = find_project_root()

LSTM_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_expanded_es"
TRANS_DIR = PROJECT_ROOT / "models" / "transformer_hurdle_v1"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LSTM_OOF_PATH = LSTM_DIR / "oof_best_epoch.parquet"
TRANS_OOF_PATH = TRANS_DIR / "oof_best_epoch.parquet"

LSTM_SUBMISSION_PATH = (
    SUBMISSION_DIR
    / "lstm_hurdle_v4_expanded_es_3seed.csv"
)

TRANS_SUBMISSION_PATH = (
    SUBMISSION_DIR
    / "transformer_hurdle_v1_3seed.csv"
)

LSTM_COMPONENTS_PATH = (
    LSTM_DIR
    / "february_seed_components.parquet"
)

TRANS_SEEDS = [42, 143, 2026]

PUBLIC_LSTM = 1.6506631932
PUBLIC_TRANSFORMER = 1.6509854083

FINAL_PATH = (
    SUBMISSION_DIR
    / "lstm_transformer_finalshot.csv"
)

print("project root:", PROJECT_ROOT)
print("final path:", FINAL_PATH)

project root: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026
final path: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/submissions/lstm_transformer_finalshot.csv


## 1. Загружаем exact OOF двух моделей

Merge key:

```text
user_id x cutoff_date
```

После merge проверяется, что `y_true` совпадает.

Для meta-analysis используются:

```text
pred_log
gate_prob
positive_log
direct_log
hurdle_log
```

Но главный FinalShot умеет работать только по `pred_log`.

### Почему OOF нужен, если public endpoints уже известны

Public дает нам очень сильный сигнал на February, но только две aggregate цифры.

OOF отвечает на другой вопрос:

> можно ли безопасно делать blend более сложным, чем один глобальный weight?

То есть leaderboard задает baseline, temporal OOF разрешает или запрещает дополнительную адаптивность.

In [6]:
COMPONENTS = [
    "pred_log",
    "gate_prob",
    "positive_log",
    "direct_log",
    "hurdle_log",
]

for path in [
    LSTM_OOF_PATH,
    TRANS_OOF_PATH,
    LSTM_SUBMISSION_PATH,
    TRANS_SUBMISSION_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)


lstm = pd.read_parquet(
    LSTM_OOF_PATH
)

trans = pd.read_parquet(
    TRANS_OOF_PATH
)

required = {
    "user_id",
    "cutoff_date",
    "y_true",
    *COMPONENTS,
}

missing_l = required - set(
    lstm.columns
)

missing_t = required - set(
    trans.columns
)

if missing_l:
    raise RuntimeError(
        f"LSTM OOF missing: {sorted(missing_l)}"
    )

if missing_t:
    raise RuntimeError(
        f"Transformer OOF missing: {sorted(missing_t)}"
    )


lstm = lstm[
    [
        "user_id",
        "cutoff_date",
        "y_true",
        *COMPONENTS,
    ]
].copy()

trans = trans[
    [
        "user_id",
        "cutoff_date",
        "y_true",
        *COMPONENTS,
    ]
].copy()


lstm = lstm.rename(
    columns={
        col: f"l_{col}"
        for col in COMPONENTS
    }
)

trans = trans.rename(
    columns={
        col: f"t_{col}"
        for col in COMPONENTS
    }
)


oof = lstm.merge(
    trans,
    on=[
        "user_id",
        "cutoff_date",
    ],
    how="inner",
    suffixes=("_l", "_t"),
    validate="one_to_one",
)


target_delta = np.max(
    np.abs(
        oof["y_true_l"].to_numpy(
            dtype=np.float64
        )
        - oof["y_true_t"].to_numpy(
            dtype=np.float64
        )
    )
)

if target_delta > 1e-5:
    raise RuntimeError(
        f"OOF targets differ: {target_delta}"
    )


oof["y_true"] = oof[
    "y_true_l"
]

oof = oof.drop(
    columns=[
        "y_true_l",
        "y_true_t",
    ]
)

oof["target_log"] = np.log1p(
    oof["y_true"].to_numpy(
        dtype=np.float64
    )
)

oof["l_pred_log"] = np.clip(
    oof["l_pred_log"].to_numpy(
        dtype=np.float64
    ),
    0,
    None,
)

oof["t_pred_log"] = np.clip(
    oof["t_pred_log"].to_numpy(
        dtype=np.float64
    ),
    0,
    None,
)


cutoffs = sorted(
    oof["cutoff_date"]
    .astype(str)
    .unique()
)

print("OOF rows:", len(oof))
print("cutoffs:", cutoffs)
print(
    "rows by cutoff:",
    oof[
        "cutoff_date"
    ].value_counts().sort_index().to_dict(),
)

OOF rows: 744983
cutoffs: ['2025-11-15', '2025-12-15', '2026-01-14']
rows by cutoff: {'2025-11-15': 244983, '2025-12-15': 250000, '2026-01-14': 250000}


## 2. February predictions и disagreement

Загружаются **ровно те submissions, у которых известны public scores**.

Это критично.

Не использовать:

```text
..._3seed_c.csv
proxy_gate
safe85
другую LSTM версию
```

Public endpoint `1.6506631932` относится именно к:

```text
lstm_hurdle_v4_expanded_es_3seed.csv
```

Transformer endpoint `1.6509854083` -- к:

```text
transformer_hurdle_v1_3seed.csv
```

Оба `predict` переводятся обратно:

$$
z=\log(1+predict).
$$

In [7]:
l_sub = pd.read_csv(
    LSTM_SUBMISSION_PATH
)

t_sub = pd.read_csv(
    TRANS_SUBMISSION_PATH
)

for name, frame in [
    ("LSTM", l_sub),
    ("Transformer", t_sub),
]:
    if not {
        "user_id",
        "predict",
    }.issubset(frame.columns):
        raise RuntimeError(
            f"{name} submission has wrong columns"
        )

    if frame["user_id"].duplicated().any():
        raise RuntimeError(
            f"{name} submission has duplicate user_id"
        )

    if not np.isfinite(
        frame["predict"]
    ).all():
        raise RuntimeError(
            f"{name} submission has non-finite predictions"
        )


final = (
    l_sub[
        ["user_id", "predict"]
    ]
    .rename(
        columns={
            "predict": "l_predict"
        }
    )
    .merge(
        t_sub[
            ["user_id", "predict"]
        ].rename(
            columns={
                "predict": "t_predict"
            }
        ),
        on="user_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(final) != len(l_sub):
    raise RuntimeError(
        "Final submissions have different user sets"
    )


final["l_pred_log"] = np.log1p(
    final["l_predict"].to_numpy(
        dtype=np.float64
    )
)

final["t_pred_log"] = np.log1p(
    final["t_predict"].to_numpy(
        dtype=np.float64
    )
)


l_final = final[
    "l_pred_log"
].to_numpy(dtype=np.float64)

t_final = final[
    "t_pred_log"
].to_numpy(dtype=np.float64)

d_final = (
    t_final
    - l_final
)

D_TEST = float(
    np.mean(
        d_final ** 2
    )
)

print("final rows:", len(final))
print(
    "pred_log correlation:",
    f"{np.corrcoef(l_final, t_final)[0,1]:.6f}",
)
print(
    "mean |T-L|:",
    f"{np.mean(np.abs(d_final)):.6f}",
)
print(
    "RMS disagreement:",
    f"{np.sqrt(D_TEST):.6f}",
)

final rows: 250000
pred_log correlation: 0.999248
mean |T-L|: 0.044956
RMS disagreement: 0.061958


## 3. Ключевой трюк -- public-informed optimal blend

Это главный кандидат notebook.

Пусть:

```text
L = log-prediction LSTM
T = log-prediction Transformer
d = T - L
```

Blend:

$$
P(w)=L+wd.
$$

На **той же public выборке** squared RMSLE вдоль этой линии:

$$
R^2(w)=A+2cw+w^2D,
$$

где:

$$
A=R_L^2,
$$

$$
D=mean((T-L)^2).
$$

А в точке `w=1`:

$$
B=R_T^2=A+2c+D.
$$

Значит:

$$
c=\frac{B-A-D}{2}.
$$

И optimum:

$$
w^*=
\frac{
A+D-B
}{
2D
}.
$$

### Что мы знаем

Мы знаем **точные public endpoint RMSLE**:

```text
R_L = 1.6506631932
R_T = 1.6509854083
```

Но public subset hidden, поэтому точный `D_public` неизвестен.

Мы используем:

```text
D_TEST = mean((T-L)^2)
```

по всем 250k February users как proxy.

Если leaderboard public/private split пользователей достаточно репрезентативен, это намного информативнее, чем выбирать weight только по старым месяцам.

### Почему это не "перефит на leaderboard"

Мы используем две уже существующие endpoint submission и строим **одну одномерную интерполяцию**.

Никакого перебора десятков leaderboard weights не происходит.

In [8]:
A_PUBLIC = PUBLIC_LSTM ** 2
B_PUBLIC = PUBLIC_TRANSFORMER ** 2


def public_proxy_weight(D):
    if D <= 1e-12:
        return 0.0

    w = (
        A_PUBLIC
        + D
        - B_PUBLIC
    ) / (
        2.0 * D
    )

    return float(
        np.clip(
            w,
            0.0,
            1.0,
        )
    )


def public_proxy_score(w, D=D_TEST):
    c = (
        B_PUBLIC
        - A_PUBLIC
        - D
    ) / 2.0

    score2 = (
        A_PUBLIC
        + 2.0 * c * w
        + D * w * w
    )

    return float(
        np.sqrt(
            max(
                score2,
                0.0,
            )
        )
    )


W_PUBLIC = public_proxy_weight(
    D_TEST
)

PUBLIC_PROXY_MIN = public_proxy_score(
    W_PUBLIC
)

print(
    "public-informed w Transformer:",
    f"{W_PUBLIC:.5f}",
)

print(
    "proxy public score at optimum:",
    f"{PUBLIC_PROXY_MIN:.9f}",
)

print(
    "proxy gain vs LSTM:",
    f"{PUBLIC_LSTM - PUBLIC_PROXY_MIN:+.9f}",
)

print()
print("Local curve around optimum:")

for delta in [
    -0.20,
    -0.10,
    -0.05,
    0.00,
    0.05,
    0.10,
    0.20,
]:
    w = float(
        np.clip(
            W_PUBLIC + delta,
            0.0,
            1.0,
        )
    )

    print(
        f"wT={w:.4f}",
        f"proxy={public_proxy_score(w):.9f}",
    )

public-informed w Transformer: 0.36144
proxy public score at optimum: 1.650511281
proxy gain vs LSTM: +0.000151912

Local curve around optimum:
wT=0.1614 proxy=1.650557797
wT=0.2614 proxy=1.650522910
wT=0.3114 proxy=1.650514188
wT=0.3614 proxy=1.650511281
wT=0.4114 proxy=1.650514188
wT=0.4614 proxy=1.650522910
wT=0.5614 proxy=1.650557797


## 4. OOF diagnostics -- насколько public-informed weight выглядит разумно

Теперь public-informed `W_PUBLIC` **не переоптимизируется** по OOF.

OOF используется как sanity check.

Для каждого месяца считаем:

- LSTM RMSLE;
- Transformer RMSLE;
- public-informed blend RMSLE;
- оптимальный weight этого месяца;
- residual correlation;
- disagreement.

Если `W_PUBLIC` дает катастрофический проигрыш на temporal folds, это сигнал сделать shrink к LSTM.

Если он находится в адекватном диапазоне и ошибки моделей не полностью коррелированы -- оставляем.

In [9]:
def rmsle_log(
    y_log,
    pred_log,
):
    return float(
        np.sqrt(
            np.mean(
                (
                    y_log
                    - np.clip(
                        pred_log,
                        0,
                        None,
                    )
                ) ** 2
            )
        )
    )


def optimal_weight(
    y,
    l,
    t,
):
    d = t - l
    den = float(
        np.dot(d, d)
    )

    if den <= 1e-12:
        return 0.0

    w = float(
        np.dot(
            d,
            y - l,
        ) / den
    )

    return float(
        np.clip(
            w,
            0.0,
            1.0,
        )
    )


diag_rows = []

for cutoff, part in oof.groupby(
    "cutoff_date"
):
    y = part[
        "target_log"
    ].to_numpy(dtype=np.float64)

    l = part[
        "l_pred_log"
    ].to_numpy(dtype=np.float64)

    t = part[
        "t_pred_log"
    ].to_numpy(dtype=np.float64)

    blend = (
        (1.0 - W_PUBLIC) * l
        + W_PUBLIC * t
    )

    diag_rows.append({
        "cutoff": str(cutoff),
        "rows": len(part),
        "lstm": rmsle_log(y, l),
        "transformer": rmsle_log(y, t),
        "public_blend": rmsle_log(
            y,
            blend,
        ),
        "delta_blend_vs_lstm": (
            rmsle_log(y, blend)
            - rmsle_log(y, l)
        ),
        "fold_optimal_w": optimal_weight(
            y,
            l,
            t,
        ),
        "prediction_corr": float(
            np.corrcoef(
                l,
                t,
            )[0,1]
        ),
        "residual_corr": float(
            np.corrcoef(
                l - y,
                t - y,
            )[0,1]
        ),
        "rms_disagreement": float(
            np.sqrt(
                np.mean(
                    (t - l) ** 2
                )
            )
        ),
    })


diagnostics = pd.DataFrame(
    diag_rows
)

display(
    diagnostics
)


W_FOLD_MEDIAN = float(
    diagnostics[
        "fold_optimal_w"
    ].median()
)

W_OOF = optimal_weight(
    oof[
        "target_log"
    ].to_numpy(dtype=np.float64),
    oof[
        "l_pred_log"
    ].to_numpy(dtype=np.float64),
    oof[
        "t_pred_log"
    ].to_numpy(dtype=np.float64),
)

W_RECENT = optimal_weight(
    oof[
        oof["cutoff_date"]
        .astype(str)
        .isin(
            cutoffs[-2:]
        )
    ]["target_log"].to_numpy(dtype=np.float64),
    oof[
        oof["cutoff_date"]
        .astype(str)
        .isin(
            cutoffs[-2:]
        )
    ]["l_pred_log"].to_numpy(dtype=np.float64),
    oof[
        oof["cutoff_date"]
        .astype(str)
        .isin(
            cutoffs[-2:]
        )
    ]["t_pred_log"].to_numpy(dtype=np.float64),
)

print("W_PUBLIC:", f"{W_PUBLIC:.4f}")
print("W_OOF:", f"{W_OOF:.4f}")
print("W_RECENT:", f"{W_RECENT:.4f}")
print(
    "W_FOLD_MEDIAN:",
    f"{W_FOLD_MEDIAN:.4f}",
)

,cutoff,rows,lstm,transformer,public_blend,delta_blend_vs_lstm,fold_optimal_w,prediction_corr,residual_corr,rms_disagreement
0,2025-11-15,244983,1.732430,1.732327,1.731937,-0.000493,0.526232,0.998828,0.998946,0.082683
1,2025-12-15,250000,1.741325,1.740965,1.740430,-0.000894,0.554380,0.998511,0.998130,0.107368
2,2026-01-14,250000,1.671569,1.672047,1.671077,-0.000492,0.417024,0.998082,0.998299,0.098131


W_PUBLIC: 0.3614
W_OOF: 0.5001
W_RECENT: 0.4919
W_FOLD_MEDIAN: 0.5262


## 5. Robust shrink -- страховка от hidden public subset

`D_TEST` считается по всем 250k test users, а public score может считаться только по hidden subset.

Поэтому точный public quadratic может немного сдвинуться.

Хорошая новость:

$$
R^2(w)-R^2(w^*)
=
D(w-w^*)^2.
$$

То есть если LSTM/Transformer очень похожи (`D` маленький), ошибка в weight сама по себе стоит мало.

Тем не менее строится **один robust global weight**.

Логика:

```text
1. W_PUBLIC -- главный сигнал.
2. Если он ухудшает любой temporal fold > 0.001:
   shrink к LSTM.
3. Если temporal optimal weights очень нестабильны:
   дополнительный shrink.
4. Иначе оставляем почти весь public-informed weight.
```

Это не отдельный submission candidate -- это автоматическая страховка внутри FinalShot.

In [10]:
worst_temporal_delta = float(
    diagnostics[
        "delta_blend_vs_lstm"
    ].max()
)

fold_weight_std = float(
    diagnostics[
        "fold_optimal_w"
    ].std(ddof=0)
)

shrink = 1.0


# Сильный historical contradiction.
if worst_temporal_delta > 0.0020:
    shrink *= 0.50

elif worst_temporal_delta > 0.0010:
    shrink *= 0.70

elif worst_temporal_delta > 0.0005:
    shrink *= 0.85


# Сильная нестабильность optimal weights между месяцами.
if fold_weight_std > 0.35:
    shrink *= 0.75

elif fold_weight_std > 0.20:
    shrink *= 0.90


W_GLOBAL_ROBUST = float(
    np.clip(
        shrink * W_PUBLIC,
        0.0,
        1.0,
    )
)


print(
    "worst temporal delta vs LSTM:",
    f"{worst_temporal_delta:+.6f}",
)

print(
    "fold optimal weight std:",
    f"{fold_weight_std:.4f}",
)

print(
    "automatic shrink:",
    f"{shrink:.4f}",
)

print(
    "W_GLOBAL_ROBUST:",
    f"{W_GLOBAL_ROBUST:.5f}",
)

print(
    "proxy public score robust:",
    f"{public_proxy_score(W_GLOBAL_ROBUST):.9f}",
)

worst temporal delta vs LSTM: -0.000492
fold optimal weight std: 0.0592
automatic shrink: 1.0000
W_GLOBAL_ROBUST: 0.36144
proxy public score robust: 1.650511281


## 6. Expanding meta-CV для adaptive candidates

Теперь проверяем, можно ли **уверенно** победить global blend более сложной логикой.

Meta folds:

```text
train Nov      -> validate Dec
train Nov+Dec  -> validate Jan
```

Baseline validation prediction всегда:

```text
W_GLOBAL_ROBUST
```

То есть adaptive candidate должен победить не просто LSTM, а уже сильный global FinalShot baseline.

Разрешены два кандидата:

```text
A. magnitude-aware convex
B. residual Ridge
```

Для одного финального сабмита threshold жесткий:

```text
mean gain vs global >= 0.00035 RMSLE
worst fold degradation <= 0.00010
```

Если условие не выполнено -- adaptive stack не имеет права заменить global blend.

Так мы используем дополнительную model capacity только при реально повторяемом temporal gain.

In [11]:
META_MIN_GAIN = 0.00035
META_MAX_WORST_DEGRADATION = 0.00010


# ============================================================================
# A. Magnitude-aware convex
# ============================================================================

MAG_QUANTILES = [0.50, 0.85]
MAG_SHRINKS = [0.25, 0.50, 0.75, 1.00]


def fit_magnitude(
    frame,
    shrink,
):
    y = frame[
        "target_log"
    ].to_numpy(dtype=np.float64)

    l = frame[
        "l_pred_log"
    ].to_numpy(dtype=np.float64)

    t = frame[
        "t_pred_log"
    ].to_numpy(dtype=np.float64)

    m = 0.5 * (
        l + t
    )

    thresholds = np.quantile(
        m,
        MAG_QUANTILES,
    )

    global_w = optimal_weight(
        y,
        l,
        t,
    )

    bucket = np.digitize(
        m,
        thresholds,
    )

    weights = []

    for b in range(3):
        mask = bucket == b

        if mask.sum() < 1000:
            local_w = global_w
        else:
            local_w = optimal_weight(
                y[mask],
                l[mask],
                t[mask],
            )

        w = (
            global_w
            + shrink
            * (
                local_w
                - global_w
            )
        )

        weights.append(
            float(
                np.clip(
                    w,
                    0.0,
                    1.0,
                )
            )
        )

    return {
        "thresholds": thresholds,
        "weights": np.asarray(
            weights,
            dtype=np.float64,
        ),
        "shrink": float(shrink),
    }


def predict_magnitude(
    frame,
    model,
):
    l = frame[
        "l_pred_log"
    ].to_numpy(dtype=np.float64)

    t = frame[
        "t_pred_log"
    ].to_numpy(dtype=np.float64)

    m = 0.5 * (
        l + t
    )

    bucket = np.digitize(
        m,
        model[
            "thresholds"
        ],
    )

    w = model[
        "weights"
    ][bucket]

    pred = (
        (1.0 - w) * l
        + w * t
    )

    return pred, w


mag_rows = []

for valid_idx in range(
    1,
    len(cutoffs),
):
    train_cutoffs = cutoffs[
        :valid_idx
    ]

    valid_cutoff = cutoffs[
        valid_idx
    ]

    train = oof[
        oof["cutoff_date"]
        .astype(str)
        .isin(
            train_cutoffs
        )
    ]

    valid = oof[
        oof["cutoff_date"]
        .astype(str)
        == valid_cutoff
    ]

    yv = valid[
        "target_log"
    ].to_numpy(dtype=np.float64)

    lv = valid[
        "l_pred_log"
    ].to_numpy(dtype=np.float64)

    tv = valid[
        "t_pred_log"
    ].to_numpy(dtype=np.float64)

    global_pred = (
        (1.0 - W_GLOBAL_ROBUST) * lv
        + W_GLOBAL_ROBUST * tv
    )

    global_score = rmsle_log(
        yv,
        global_pred,
    )

    for s in MAG_SHRINKS:
        model = fit_magnitude(
            train,
            shrink=s,
        )

        pred, applied_w = (
            predict_magnitude(
                valid,
                model,
            )
        )

        score = rmsle_log(
            yv,
            pred,
        )

        mag_rows.append({
            "valid_cutoff": valid_cutoff,
            "shrink": s,
            "score": score,
            "global_score": global_score,
            "delta_vs_global": (
                score
                - global_score
            ),
            "mean_w": float(
                applied_w.mean()
            ),
            "w_low": model["weights"][0],
            "w_mid": model["weights"][1],
            "w_high": model["weights"][2],
        })


mag_cv = pd.DataFrame(
    mag_rows
)

mag_summary = (
    mag_cv
    .groupby(
        "shrink",
        as_index=False,
    )
    .agg(
        mean_score=(
            "score",
            "mean",
        ),
        mean_global=(
            "global_score",
            "mean",
        ),
        mean_delta=(
            "delta_vs_global",
            "mean",
        ),
        worst_delta=(
            "delta_vs_global",
            "max",
        ),
    )
)

mag_summary[
    "gain_vs_global"
] = (
    -mag_summary[
        "mean_delta"
    ]
)

mag_summary[
    "passes"
] = (
    (
        mag_summary[
            "gain_vs_global"
        ]
        >= META_MIN_GAIN
    )
    & (
        mag_summary[
            "worst_delta"
        ]
        <= META_MAX_WORST_DEGRADATION
    )
)

display(
    mag_cv.sort_values(
        [
            "valid_cutoff",
            "score",
        ]
    )
)

display(
    mag_summary.sort_values(
        [
            "passes",
            "mean_score",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


passing_mag = (
    mag_summary[
        mag_summary[
            "passes"
        ]
    ]
    .sort_values(
        "mean_score"
    )
)


if len(passing_mag):
    MAG_SELECTED_SHRINK = float(
        passing_mag.iloc[0][
            "shrink"
        ]
    )

    MAG_PASSES = True

else:
    MAG_SELECTED_SHRINK = float(
        mag_summary
        .sort_values(
            "mean_score"
        )
        .iloc[0][
            "shrink"
        ]
    )

    MAG_PASSES = False


MAG_FINAL = fit_magnitude(
    oof,
    shrink=MAG_SELECTED_SHRINK,
)


print(
    "MAG_PASSES:",
    MAG_PASSES,
)

print(
    "selected shrink:",
    MAG_SELECTED_SHRINK,
)

print(
    "final magnitude weights:",
    MAG_FINAL["weights"],
)

,valid_cutoff,shrink,score,global_score,delta_vs_global,mean_w,w_low,w_mid,w_high
0,2025-12-15,0.25,1.740314,1.740430,-0.000116,0.527644,0.532007,0.519121,0.533551
1,2025-12-15,0.50,1.740318,1.740430,-0.000112,0.529056,0.537781,0.512010,0.540870
2,2025-12-15,0.75,1.740323,1.740430,-0.000107,0.530468,0.543555,0.504900,0.548188
3,2025-12-15,1.00,1.740328,1.740430,-0.000102,0.531880,0.549330,0.497789,0.555507
4,2026-01-14,0.25,1.671222,1.671077,0.000145,0.578171,0.635374,0.548170,0.408026
5,2026-01-14,0.50,1.671364,1.671077,0.000287,0.612307,0.726714,0.552305,0.272017
6,2026-01-14,0.75,1.671541,1.671077,0.000464,0.646444,0.818054,0.556440,0.136009
7,2026-01-14,1.00,1.671752,1.671077,0.000675,0.680580,0.909394,0.560576,0.000000


,shrink,mean_score,mean_global,mean_delta,worst_delta,gain_vs_global,passes
0,0.25,1.705768,1.705754,0.000014,0.000145,-0.000014,False
1,0.50,1.705841,1.705754,0.000088,0.000287,-0.000088,False
2,0.75,1.705932,1.705754,0.000178,0.000464,-0.000178,False
3,1.00,1.706040,1.705754,0.000287,0.000675,-0.000287,False


MAG_PASSES: False
selected shrink: 0.25
final magnitude weights: [0.54745814 0.50578247 0.42679637]


## 7. Residual Ridge -- только маленькая correction к global blend

Ridge запускается только если February components есть у обеих моделей.

Target для meta-model:

$$
r =
y_{log}
-
global\_blend_{log}.
$$

То есть Ridge **не строит новый prediction с нуля**.

Он может только поправить уже сильный public-informed blend.

Meta features:

```text
T_pred - L_pred
|T_pred - L_pred|

T_gate - L_gate
T_positive - L_positive
T_direct - L_direct
T_hurdle - L_hurdle

mean pred
mean gate
```

Final correction:

```text
shrink * Ridge(...)
```

с hard clipping.

И опять применяется тот же жесткий temporal gate:

```text
mean gain >= 0.00035
worst degradation <= 0.00010
```

In [12]:
META_FEATURES = [
    "d_pred",
    "abs_d_pred",
    "mean_pred",
    "d_gate",
    "mean_gate",
    "d_positive",
    "d_direct",
    "d_hurdle",
]


def add_meta_features(
    frame,
):
    frame = frame.copy()

    frame[
        "d_pred"
    ] = (
        frame[
            "t_pred_log"
        ]
        - frame[
            "l_pred_log"
        ]
    )

    frame[
        "abs_d_pred"
    ] = np.abs(
        frame[
            "d_pred"
        ]
    )

    frame[
        "mean_pred"
    ] = 0.5 * (
        frame[
            "t_pred_log"
        ]
        + frame[
            "l_pred_log"
        ]
    )

    frame[
        "d_gate"
    ] = (
        frame[
            "t_gate_prob"
        ]
        - frame[
            "l_gate_prob"
        ]
    )

    frame[
        "mean_gate"
    ] = 0.5 * (
        frame[
            "t_gate_prob"
        ]
        + frame[
            "l_gate_prob"
        ]
    )

    frame[
        "d_positive"
    ] = (
        frame[
            "t_positive_log"
        ]
        - frame[
            "l_positive_log"
        ]
    )

    frame[
        "d_direct"
    ] = (
        frame[
            "t_direct_log"
        ]
        - frame[
            "l_direct_log"
        ]
    )

    frame[
        "d_hurdle"
    ] = (
        frame[
            "t_hurdle_log"
        ]
        - frame[
            "l_hurdle_log"
        ]
    )

    return frame


oof_meta = add_meta_features(
    oof
)


ALPHAS = [
    1e3,
    1e4,
    1e5,
]

RIDGE_SHRINKS = [
    0.25,
    0.50,
    0.75,
]

RIDGE_CAPS = [
    0.10,
    0.20,
    0.30,
]


ridge_rows = []

for valid_idx in range(
    1,
    len(cutoffs),
):
    train_cutoffs = cutoffs[
        :valid_idx
    ]

    valid_cutoff = cutoffs[
        valid_idx
    ]

    train = oof_meta[
        oof_meta[
            "cutoff_date"
        ]
        .astype(str)
        .isin(
            train_cutoffs
        )
    ]

    valid = oof_meta[
        oof_meta[
            "cutoff_date"
        ]
        .astype(str)
        == valid_cutoff
    ]

    X_train = train[
        META_FEATURES
    ].to_numpy(
        dtype=np.float64
    )

    X_valid = valid[
        META_FEATURES
    ].to_numpy(
        dtype=np.float64
    )

    y_train = train[
        "target_log"
    ].to_numpy(
        dtype=np.float64
    )

    l_train = train[
        "l_pred_log"
    ].to_numpy(
        dtype=np.float64
    )

    t_train = train[
        "t_pred_log"
    ].to_numpy(
        dtype=np.float64
    )

    baseline_train = (
        (1.0 - W_GLOBAL_ROBUST)
        * l_train
        + W_GLOBAL_ROBUST
        * t_train
    )

    residual_target = (
        y_train
        - baseline_train
    )

    y_valid = valid[
        "target_log"
    ].to_numpy(
        dtype=np.float64
    )

    l_valid = valid[
        "l_pred_log"
    ].to_numpy(
        dtype=np.float64
    )

    t_valid = valid[
        "t_pred_log"
    ].to_numpy(
        dtype=np.float64
    )

    baseline_valid = (
        (1.0 - W_GLOBAL_ROBUST)
        * l_valid
        + W_GLOBAL_ROBUST
        * t_valid
    )

    baseline_score = rmsle_log(
        y_valid,
        baseline_valid,
    )

    for alpha in ALPHAS:
        model = Pipeline([
            (
                "scale",
                StandardScaler(),
            ),
            (
                "ridge",
                Ridge(
                    alpha=alpha
                ),
            ),
        ])

        model.fit(
            X_train,
            residual_target,
        )

        raw = model.predict(
            X_valid
        )

        for shrink in RIDGE_SHRINKS:
            for cap in RIDGE_CAPS:
                corr = np.clip(
                    shrink * raw,
                    -cap,
                    cap,
                )

                pred = np.clip(
                    baseline_valid
                    + corr,
                    0,
                    None,
                )

                score = rmsle_log(
                    y_valid,
                    pred,
                )

                ridge_rows.append({
                    "valid_cutoff": valid_cutoff,
                    "alpha": alpha,
                    "shrink": shrink,
                    "cap": cap,
                    "score": score,
                    "global_score": baseline_score,
                    "delta_vs_global": (
                        score
                        - baseline_score
                    ),
                })


ridge_cv = pd.DataFrame(
    ridge_rows
)


ridge_summary = (
    ridge_cv
    .groupby(
        [
            "alpha",
            "shrink",
            "cap",
        ],
        as_index=False,
    )
    .agg(
        mean_score=(
            "score",
            "mean",
        ),
        mean_global=(
            "global_score",
            "mean",
        ),
        mean_delta=(
            "delta_vs_global",
            "mean",
        ),
        worst_delta=(
            "delta_vs_global",
            "max",
        ),
    )
)


ridge_summary[
    "gain_vs_global"
] = (
    -ridge_summary[
        "mean_delta"
    ]
)

ridge_summary[
    "passes"
] = (
    (
        ridge_summary[
            "gain_vs_global"
        ]
        >= META_MIN_GAIN
    )
    & (
        ridge_summary[
            "worst_delta"
        ]
        <= META_MAX_WORST_DEGRADATION
    )
)


display(
    ridge_summary
    .sort_values(
        [
            "passes",
            "mean_score",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(20)
)


passing_ridge = (
    ridge_summary[
        ridge_summary[
            "passes"
        ]
    ]
    .sort_values(
        "mean_score"
    )
)


RIDGE_PASSES = (
    len(
        passing_ridge
    )
    > 0
)


if RIDGE_PASSES:
    best_ridge = (
        passing_ridge
        .iloc[0]
    )

else:
    best_ridge = (
        ridge_summary
        .sort_values(
            "mean_score"
        )
        .iloc[0]
    )


RIDGE_ALPHA = float(
    best_ridge[
        "alpha"
    ]
)

RIDGE_SHRINK = float(
    best_ridge[
        "shrink"
    ]
)

RIDGE_CAP = float(
    best_ridge[
        "cap"
    ]
)


print(
    "RIDGE_PASSES temporal gate:",
    RIDGE_PASSES,
)

print(
    "selected:",
    RIDGE_ALPHA,
    RIDGE_SHRINK,
    RIDGE_CAP,
)

,alpha,shrink,cap,mean_score,mean_global,mean_delta,worst_delta,gain_vs_global,passes
18,100000.0,0.25,0.1,1.705744,1.705754,-0.000009,0.000429,0.000009,False
19,100000.0,0.25,0.2,1.705745,1.705754,-0.000009,0.000429,0.000009,False
20,100000.0,0.25,0.3,1.705745,1.705754,-0.000009,0.000429,0.000009,False
9,10000.0,0.25,0.1,1.705787,1.705754,0.000034,0.000439,-0.000034,False
11,10000.0,0.25,0.3,1.705787,1.705754,0.000034,0.000439,-0.000034,False
10,10000.0,0.25,0.2,1.705788,1.705754,0.000034,0.000439,-0.000034,False
0,1000.0,0.25,0.1,1.705818,1.705754,0.000065,0.000433,-0.000065,False
2,1000.0,0.25,0.3,1.705819,1.705754,0.000065,0.000434,-0.000065,False
1,1000.0,0.25,0.2,1.705819,1.705754,0.000065,0.000434,-0.000065,False
21,100000.0,0.50,0.1,1.705830,1.705754,0.000077,0.000921,-0.000077,False


RIDGE_PASSES temporal gate: False
selected: 100000.0 0.25 0.1


## 8. Загружаем February components для Ridge

Если components отсутствуют:

```text
RIDGE_FINAL_AVAILABLE = False
```

и Ridge автоматически не может стать FinalShot.

Magnitude-aware/global blend components не требуют.

In [13]:
RIDGE_FINAL_AVAILABLE = False
final_meta = None


if LSTM_COMPONENTS_PATH.exists():
    l_comp_raw = pd.read_parquet(
        LSTM_COMPONENTS_PATH
    )

    needed = {
        "user_id",
        "seed",
        *COMPONENTS,
    }

    if needed.issubset(
        l_comp_raw.columns
    ):
        l_comp = (
            l_comp_raw
            .groupby(
                "user_id",
                as_index=False,
            )[
                COMPONENTS
            ]
            .mean()
            .rename(
                columns={
                    col: f"l_{col}"
                    for col in COMPONENTS
                }
            )
        )

    else:
        l_comp = None

else:
    l_comp = None


t_seed_frames = []

for seed in TRANS_SEEDS:
    path = (
        TRANS_DIR
        / f"inference_seed_{seed}.npz"
    )

    if not path.exists():
        t_seed_frames = []
        break

    saved = np.load(
        path
    )

    if not set(
        COMPONENTS
    ).issubset(
        saved.files
    ):
        t_seed_frames = []
        break

    t_seed_frames.append(
        pd.DataFrame({
            "user_id": saved[
                "user_id"
            ],
            **{
                col: saved[
                    col
                ]
                for col in COMPONENTS
            },
        })
    )


if (
    l_comp is not None
    and t_seed_frames
):
    t_comp = (
        pd.concat(
            t_seed_frames,
            ignore_index=True,
        )
        .groupby(
            "user_id",
            as_index=False,
        )[
            COMPONENTS
        ]
        .mean()
        .rename(
            columns={
                col: f"t_{col}"
                for col in COMPONENTS
            }
        )
    )

    final_meta = (
        l_comp
        .merge(
            t_comp,
            on="user_id",
            how="inner",
            validate="one_to_one",
        )
    )

    final_meta = add_meta_features(
        final_meta
    )

    RIDGE_FINAL_AVAILABLE = (
        len(final_meta)
        == len(final)
    )


print(
    "RIDGE_FINAL_AVAILABLE:",
    RIDGE_FINAL_AVAILABLE,
)

RIDGE_FINAL_AVAILABLE: False


## 9. Автоматический выбор ОДНОГО FinalShot

Baseline:

```text
public-informed robust global blend
```

Magnitude-aware заменяет baseline только если прошел temporal gate.

Ridge заменяет baseline только если:

```text
1. прошел temporal gate;
2. February components доступны;
3. его mean meta-CV лучше magnitude candidate, если magnitude тоже прошел.
```

То есть финальный selector не выбирает по full-OOF minimum.

Он выбирает только между кандидатами, которые показали **повторяемый expanding temporal gain относительно уже сильного public-informed baseline**.

Если ни один adaptive candidate не доказал gain -- используем global blend.

Это именно то поведение, которое нужно при одном оставшемся submission.

In [14]:
# ============================================================================
# Build global baseline
# ============================================================================

global_final_log = (
    (1.0 - W_GLOBAL_ROBUST)
    * l_final
    + W_GLOBAL_ROBUST
    * t_final
)


selected_name = (
    "public_informed_global"
)

selected_pred_log = (
    global_final_log.copy()
)

selected_meta_mean_gain = 0.0


# ============================================================================
# Magnitude candidate
# ============================================================================

if MAG_PASSES:
    final_mag_frame = pd.DataFrame({
        "l_pred_log": l_final,
        "t_pred_log": t_final,
    })

    mag_final_log, mag_final_weights = (
        predict_magnitude(
            final_mag_frame,
            MAG_FINAL,
        )
    )

    mag_best_row = (
        passing_mag
        .iloc[0]
    )

    mag_gain = float(
        mag_best_row[
            "gain_vs_global"
        ]
    )

    if (
        mag_gain
        > selected_meta_mean_gain
    ):
        selected_name = (
            "magnitude_aware"
        )

        selected_pred_log = (
            mag_final_log
        )

        selected_meta_mean_gain = (
            mag_gain
        )


# ============================================================================
# Ridge candidate
# ============================================================================

if (
    RIDGE_PASSES
    and RIDGE_FINAL_AVAILABLE
):
    X_all = oof_meta[
        META_FEATURES
    ].to_numpy(
        dtype=np.float64
    )

    y_all = oof_meta[
        "target_log"
    ].to_numpy(
        dtype=np.float64
    )

    l_all_meta = oof_meta[
        "l_pred_log"
    ].to_numpy(
        dtype=np.float64
    )

    t_all_meta = oof_meta[
        "t_pred_log"
    ].to_numpy(
        dtype=np.float64
    )

    baseline_all = (
        (1.0 - W_GLOBAL_ROBUST)
        * l_all_meta
        + W_GLOBAL_ROBUST
        * t_all_meta
    )

    residual_all = (
        y_all
        - baseline_all
    )

    ridge_final = Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(
                alpha=RIDGE_ALPHA
            ),
        ),
    ])

    ridge_final.fit(
        X_all,
        residual_all,
    )

    raw_corr = ridge_final.predict(
        final_meta[
            META_FEATURES
        ].to_numpy(
            dtype=np.float64
        )
    )

    correction = np.clip(
        RIDGE_SHRINK
        * raw_corr,
        -RIDGE_CAP,
        RIDGE_CAP,
    )

    # final_meta user order может отличаться -- делаем map.
    ridge_log_by_user = pd.Series(
        (
            (1.0 - W_GLOBAL_ROBUST)
            * final_meta[
                "l_pred_log"
            ].to_numpy(
                dtype=np.float64
            )
            + W_GLOBAL_ROBUST
            * final_meta[
                "t_pred_log"
            ].to_numpy(
                dtype=np.float64
            )
            + correction
        ),
        index=final_meta[
            "user_id"
        ],
    )

    ridge_final_log = (
        final[
            "user_id"
        ]
        .map(
            ridge_log_by_user
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    if not np.isfinite(
        ridge_final_log
    ).all():
        raise RuntimeError(
            "Ridge final mapping failed"
        )

    ridge_gain = float(
        best_ridge[
            "gain_vs_global"
        ]
    )

    if (
        ridge_gain
        > selected_meta_mean_gain
    ):
        selected_name = (
            "residual_ridge"
        )

        selected_pred_log = (
            np.clip(
                ridge_final_log,
                0,
                None,
            )
        )

        selected_meta_mean_gain = (
            ridge_gain
        )


print("=" * 90)
print("FINAL SHOT SELECTION")
print("=" * 90)

print(
    "selected:",
    selected_name,
)

print(
    "W_PUBLIC:",
    f"{W_PUBLIC:.5f}",
)

print(
    "W_GLOBAL_ROBUST:",
    f"{W_GLOBAL_ROBUST:.5f}",
)

print(
    "MAG_PASSES:",
    MAG_PASSES,
)

print(
    "RIDGE_PASSES:",
    RIDGE_PASSES,
)

print(
    "RIDGE_FINAL_AVAILABLE:",
    RIDGE_FINAL_AVAILABLE,
)

print(
    "selected meta mean gain vs global:",
    f"{selected_meta_mean_gain:+.6f}",
)

FINAL SHOT SELECTION
selected: public_informed_global
W_PUBLIC: 0.36144
W_GLOBAL_ROBUST: 0.36144
MAG_PASSES: False
RIDGE_PASSES: False
RIDGE_FINAL_AVAILABLE: False
selected meta mean gain vs global: +0.000000


## 10. Сохраняем ровно один submission

Независимо от выбранного метода:

```text
user_id,predict
```

`predict` восстанавливается:

$$
predict=\exp(pred\_log)-1.
$$

Файл:

```text
submissions/lstm_transformer_finalshot.csv
```

Дополнительно сохраняется маленький `.txt` report с тем, что именно было выбрано.

Никакие alternative CSV notebook не создает.

In [15]:
selected_pred_log = np.clip(
    np.asarray(
        selected_pred_log,
        dtype=np.float64,
    ),
    0,
    None,
)

predict = np.expm1(
    selected_pred_log
)


submission = pd.DataFrame({
    "user_id": final[
        "user_id"
    ],
    "predict": predict,
})


if len(submission) != len(l_sub):
    raise RuntimeError(
        "Wrong submission length"
    )

if submission[
    "user_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate user_id"
    )

if not np.isfinite(
    submission[
        "predict"
    ]
).all():
    raise RuntimeError(
        "Non-finite predictions"
    )

if (
    submission[
        "predict"
    ]
    < 0
).any():
    raise RuntimeError(
        "Negative predictions"
    )


submission.to_csv(
    FINAL_PATH,
    index=False,
)


report_path = (
    FINAL_PATH
    .with_suffix(
        ".txt"
    )
)

report = f"""
selected={selected_name}
public_lstm={PUBLIC_LSTM:.10f}
public_transformer={PUBLIC_TRANSFORMER:.10f}
D_test={D_TEST:.10f}
W_public={W_PUBLIC:.8f}
W_global_robust={W_GLOBAL_ROBUST:.8f}
proxy_public_global={public_proxy_score(W_GLOBAL_ROBUST):.10f}
magnitude_passes={MAG_PASSES}
ridge_passes={RIDGE_PASSES}
ridge_final_available={RIDGE_FINAL_AVAILABLE}
selected_meta_mean_gain_vs_global={selected_meta_mean_gain:.8f}
rows={len(submission)}
""".strip()

report_path.write_text(
    report,
    encoding="utf-8",
)


print("SAVED:", FINAL_PATH)
print("REPORT:", report_path)
print()
print(report)

display(
    submission.head()
)

SAVED: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/submissions/lstm_transformer_finalshot.csv
REPORT: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/submissions/lstm_transformer_finalshot.txt

selected=public_informed_global
public_lstm=1.6506631932
public_transformer=1.6509854083
D_test=0.0038388189
W_public=0.36143641
W_global_robust=0.36143641
proxy_public_global=1.6505112809
magnitude_passes=False
ridge_passes=False
ridge_final_available=False
selected_meta_mean_gain_vs_global=0.00000000
rows=250000


,user_id,predict
0,2,1.768050
1,7,80.652007
2,15,6.579077
3,18,127.932072
4,23,0.365829


# Как читать результат перед отправкой

У тебя один submission, поэтому в идеале **ничего руками уже не выбирать**.

Notebook напечатает:

```text
selected=...
W_public=...
W_global_robust=...
proxy_public_global=...
```

И сохранит один:

```text
lstm_transformer_finalshot.csv
```

## Что я считаю наиболее вероятным исходом

С учетом public:

```text
LSTM        1.650663
Transformer 1.650985
```

и того, что модели довольно близки по architecture/task formulation, наиболее вероятный победитель -- **не агрессивный Ridge**, а global/mildly-adaptive log-space blend.

Но notebook даст magnitude/Ridge шанс только если они действительно улучшают оба expanding meta-fold достаточно сильно.

Это оптимизация не максимального локального OOF, а **expected leaderboard score при одном оставшемся выстреле**.